In [14]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from math import sqrt, isfinite
from scipy.stats import t as t_dist, spearmanr, norm, pearsonr

In [8]:
# Pandas :: view ALL
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

In [10]:
# Pandas :: Default view
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

In [3]:
# Defines
def analyse_dataframe(df):
    nan_cnt = df.isnull().sum()
    nan_rto = df.isnull().mean()*100
    unique_vals_cnt = df.nunique()

    ana_result = pd.DataFrame({
        'NaN Counts' : nan_cnt
        , 'NaN Ratio (%)' : nan_rto
        , 'Unique Values' : unique_vals_cnt
    })
    return ana_result


In [4]:
## PRE
out = r"D:\JHAhn\files"

In [34]:
df = pd.read_excel(os.path.join(out, "Auto_ALPS_index_19-24_main.xlsx"))
df

,PDI,PID,DDX,Group,Age,Sex_01,Sex,Education (Year),MMSE,CDR,...,LT_Entorhinal_CTX,RT_Entorhinal_CTX,R_LT_insulaHFC-HippoVol,R_RT_insulaHFC-HippoVol,R_LT_CPV-HippoVol,R_RT_CPV-HippoVol,R_LT_insulaEC-HippoVol,R_RT_insulaEC-HippoVol,R_LT_insulaIC-HippoVol,R_RT_insulaIC-HippoVol
0,EPT2022AD40_RKJ_NODDI32,9463703,NL,0,74,0,M,18,29,0.0,...,0.189083,0.167442,1.642874,1.628935,0.281551,0.224492,1.356123,1.485267,0.348935,0.186446
1,EPT2023AD09_LSU_NODDI32,8787368,NL,0,81,0,M,NaN,30,0.5,...,0.114483,0.142610,2.724480,2.788723,0.390487,0.298719,2.227149,2.499115,0.566516,0.363421
2,MWI_CN01_KNH_NODDI32,145616,NL,0,70,1,F,16,30,0.0,...,0.188636,0.152508,1.601043,1.499597,0.171926,0.164455,1.316706,1.392687,0.433419,0.246285
3,MWI_CN04_SSS_NODDI32,2716077,NL,0,62,1,F,12,30,0.0,...,0.237150,0.215680,1.112680,1.247845,0.110682,0.152231,0.936554,1.133000,0.268350,0.207276
4,MWI_CN05_LSS_NODDI32,469852,NL,0,72,0,M,16,28,0.0,...,0.182624,0.175534,1.616674,1.523099,0.223684,0.181004,1.251258,1.390741,0.388599,0.188881
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,MREPT2020AD03_KJH_NODDI32,4717899,AD,2,60,0,M,14,24,1.0,...,0.122341,0.116344,3.177561,3.409091,0.308453,0.301549,2.637849,3.138021,0.688831,0.380781
98,MREPT2020AD08_CYS_NODDI32,2933875,AD,2,73,1,F,10,20,1.0,...,0.083406,0.090051,3.708776,3.386145,0.349458,0.315932,3.026115,2.998850,0.865564,0.332676
99,MREPT2020AD12_KOB_NODDI32,8542934,AD,2,81,1,F,7,20,1.0,...,0.141278,0.139813,2.633942,2.858293,0.234101,0.250041,2.186676,2.692549,0.622930,0.339742
100,MREPT2020AD15_LHS_NODDI32,3076738,AD,2,75,1,F,2,22,1.0,...,0.175587,0.151738,1.677604,1.752083,0.230778,0.192716,1.294007,1.622849,0.499275,0.258289


In [35]:
analyse_dataframe(df)

,NaN Counts,NaN Ratio (%),Unique Values
PDI,1,0.980392,99
PID,0,0.000000,101
DDX,0,0.000000,4
Group,0,0.000000,3
Age,0,0.000000,30
...,...,...,...
R_RT_CPV-HippoVol,0,0.000000,101
R_LT_insulaEC-HippoVol,0,0.000000,102
R_RT_insulaEC-HippoVol,0,0.000000,102
R_LT_insulaIC-HippoVol,0,0.000000,102


In [36]:
source = os.path.join(out, "Auto_ALPS_index_19-24_main.xlsx")

X_cols = [
    "LT_ALL_ALPS index","LT_b800_ALPS index","LT_b2000_ALPS index",
    "RT_ALL_ALPS index","RT_b800_ALPS index","RT_b2000_ALPS index"
]

Y_prefixes = ("HFC_", "EC_", "IC_")
Y_explicit = [
    "LT_Thalamus","LT_Hippocampus","LT_Amygdala","LT_ChoroidPlexus",
    "RT_Thalamus","RT_Hippocampus","RT_Amygdala","RT_ChoroidPlexus",
    "CorpusCallosum","Pos_CC","MidPos_CC","Central_CC","MidAnt_CC","Ant_CC",
    "LT_MiddleTemporal_CTX","LT_ParaHippocampal_CTX","LT_Precuneus_CTX","LT_Insula_CTX",
    "RT_MiddleTemporal_CTX","RT_ParaHippocampal_CTX","RT_Precuneus_CTX","RT_Insula_CTX",
    "LT_Lat_Ventricle","CSF","RT_Lat_Ventricle","LT_Entorhinal_CTX","RT_Entorhinal_CTX",
    "R_LT_insulaHFC-HippoVol","R_RT_insulaHFC-HippoVol","R_LT_CPV-HippoVol","R_RT_CPV-HippoVol",
    "R_LT_insulaEC-HippoVol","R_RT_insulaEC-HippoVol","R_LT_insulaIC-HippoVol","R_RT_insulaIC-HippoVol"
]
covar = "Age"

In [37]:
def residualise(y: np.ndarray, A: np.ndarray) -> np.ndarray:
    """ y ~ A 선형회귀 잔차 (상수항 포함) """
    X = np.column_stack([np.ones(len(A)), A])  # [1, Age]
    beta, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    return y - X @ beta

# ===== 로드 & 선택 =====
df = pd.read_excel(source)

# Y 후보 선택: 접두사 + 명시 리스트
Y_cols = [c for c in df.columns if c.startswith(Y_prefixes)]
for c in Y_explicit:
    if c in df.columns and c not in Y_cols:
        Y_cols.append(c)

# 숫자형 변환 (필요 열만)
need_cols = list(set(X_cols + Y_cols + [covar]))
df_num = df[need_cols].apply(pd.to_numeric, errors="coerce")

# ===== 분석 =====
results = {}  # {X: DataFrame}

for x in X_cols:
    rows = []
    for y in Y_cols:
        sub = df_num[[x, y, covar]].dropna()
        sub = sub[(sub[x] > 0) & (sub[y] > 0)]
        if len(sub) < 3:
            rows.append({"Variable_X": x, "Variable_Y": y, "r": np.nan, "p": np.nan, "n": len(sub)})
            continue

        x_ln = np.log(sub[x].to_numpy(dtype=float))
        y_ln = np.log(sub[y].to_numpy(dtype=float))
        a    = sub[covar].to_numpy(dtype=float)

        rx = residualise(x_ln, a)
        ry = residualise(y_ln, a)

        if np.std(rx) == 0 or np.std(ry) == 0:
            r, p = np.nan, np.nan
        else:
            r, p = pearsonr(rx, ry)

        # r은 소수점 4자리, p는 규칙 적용
        if np.isnan(p):
            p_out = np.nan
        elif p < 0.001:
            p_out = "≤0.001"
        else:
            p_out = f"{p:.3f}"

        rows.append({
            "Variable_X": x,
            "Variable_Y": y,
            "r": round(r, 3) if not np.isnan(r) else np.nan,
            "p": p_out,
            "n": len(sub)
        })

    res_df = pd.DataFrame(rows).sort_values(["Variable_Y"]).reset_index(drop=True)
    results[x] = res_df

Y_order = [
    "HFC_LT_Amygdala", "EC_LT_Amygdala", "IC_LT_Amygdala"
    , "HFC_LT_Hippocampus", "EC_LT_Hippocampus", "IC_LT_Hippocampus"
    , "HFC_LT_Insula", "EC_LT_Insula", "IC_LT_Insula"
    , "HFC_LT_MTG", "EC_LT_MTG", "IC_LT_MTG"
    , "HFC_LT_ParaHippocampus", "EC_LT_ParaHippocampus", "IC_LT_ParaHippocampus"
    , "HFC_LT_Thalamus", "EC_LT_Thalamus", "IC_LT_Thalamus"
    , "HFC_RT_Amygdala", "EC_RT_Amygdala", "IC_RT_Amygdala"
    , "HFC_RT_Hippocampus", "EC_RT_Hippocampus", "IC_RT_Hippocampus"
    , "HFC_RT_Insula", "EC_RT_Insula", "IC_RT_Insula"
    , "HFC_RT_MTG", "EC_RT_MTG", "IC_RT_MTG"
    , "HFC_RT_ParaHippocampus", "EC_RT_ParaHippocampus", "IC_RT_ParaHippocampus"
    , "HFC_RT_Thalamus", "EC_RT_Thalamus", "IC_RT_Thalamus"
    , "HFC_Whole_CC", "EC_Whole_CC", "IC_Whole_CC"
    , "HFC_Whole_Precuneus", "EC_Whole_Precuneus", "IC_Whole_Precuneus"
    , "LT_Thalamus","LT_Hippocampus","LT_Amygdala","LT_ChoroidPlexus"
    , "RT_Thalamus","RT_Hippocampus","RT_Amygdala","RT_ChoroidPlexus"
    , "CorpusCallosum","Pos_CC","MidPos_CC","Central_CC","MidAnt_CC","Ant_CC"
    , "LT_MiddleTemporal_CTX","LT_ParaHippocampal_CTX","LT_Precuneus_CTX","LT_Insula_CTX"
    , "RT_MiddleTemporal_CTX","RT_ParaHippocampal_CTX","RT_Precuneus_CTX","RT_Insula_CTX"
    , "LT_Lat_Ventricle","CSF","RT_Lat_Ventricle"
    , "LT_Entorhinal_CTX","RT_Entorhinal_CTX"
    , "R_LT_insulaHFC-HippoVol","R_RT_insulaHFC-HippoVol"
    , "R_LT_CPV-HippoVol","R_RT_CPV-HippoVol"
    , "R_LT_insulaEC-HippoVol","R_RT_insulaEC-HippoVol"
    , "R_LT_insulaIC-HippoVol","R_RT_insulaIC-HippoVol"
]

# Sort order of features
for x in X_cols:
    df_res = results[x].copy()
    df_res["Variable_Y"] = pd.Categorical(df_res["Variable_Y"], categories=Y_order, ordered=True)
    df_res = df_res.sort_values("Variable_Y").reset_index(drop=True)
    results[x] = df_res

results

{'LT_ALL_ALPS index':            Variable_X              Variable_Y      r       p    n
 0   LT_ALL_ALPS index         HFC_LT_Amygdala -0.010   0.921  102
 1   LT_ALL_ALPS index          EC_LT_Amygdala -0.039   0.699  102
 2   LT_ALL_ALPS index          IC_LT_Amygdala  0.083   0.405  102
 3   LT_ALL_ALPS index      HFC_LT_Hippocampus -0.338  ≤0.001  102
 4   LT_ALL_ALPS index       EC_LT_Hippocampus -0.242   0.014  102
 ..                ...                     ...    ...     ...  ...
 72  LT_ALL_ALPS index       R_RT_CPV-HippoVol -0.299   0.002  102
 73  LT_ALL_ALPS index  R_LT_insulaEC-HippoVol -0.379  ≤0.001  102
 74  LT_ALL_ALPS index  R_RT_insulaEC-HippoVol -0.365  ≤0.001  102
 75  LT_ALL_ALPS index  R_LT_insulaIC-HippoVol -0.210   0.034  102
 76  LT_ALL_ALPS index  R_RT_insulaIC-HippoVol -0.166   0.095  102
 
 [77 rows x 5 columns],
 'LT_b800_ALPS index':             Variable_X              Variable_Y      r       p    n
 0   LT_b800_ALPS index         HFC_LT_Amygdala -0.058   0.

In [39]:
df_dest = os.path.join(out, "PartialCorr_ALPS_ln_M01.xlsx")

with pd.ExcelWriter(df_dest, engine="openpyxl") as w:
    for x in X_cols:
        results[x].to_excel(w, sheet_name=x[:31], index=False)  # 시트명 31자 제한 대응

print("Saved to -->  ", df_dest)

Saved to -->   D:\JHAhn\files\PartialCorr_ALPS_ln_M01.xlsx
